# Assignment 10 — GQA $\subset$ MLA: Proof and Implementation (100 points)

This assignment mirrors **USAAIO 2025 Round 2 Problem 2, Parts 9-13**. You will work through the complete mathematical proof chain connecting GQA to MLA, implement conversion functions, derive reduced matrices, and verify equivalence.

**Notation:**
- $B$: batch, $L$: seq length, $D$: model dim, $H$: heads, $G$: GQA groups
- $D_{qk}$: query/key dim, $D_v$: value dim, $r$: MLA latent rank
- $W^{DKV} \in \mathbb{R}^{D \times r}$: shared down-projection
- $W^{UK}_h \in \mathbb{R}^{r \times D_{qk}}$: per-head key up-projection
- $W^{UV}_h \in \mathbb{R}^{r \times D_v}$: per-head value up-projection
- Compressed cache: $C = X W^{DKV} \in \mathbb{R}^{L \times r}$
- Reduced query: $\hat{W}^Q_h = W^Q_h (W^{UK}_h)^T \in \mathbb{R}^{D \times r}$

In [ ]:
"""
DO NOT MAKE ANY CHANGE IN THIS CELL.
"""
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

**WARNING**: You may only use `torch`, `torch.nn`, `torch.nn.functional`, and `numpy`. No other imports are allowed.

## Part 1 (5 points, non-coding task)

**Setup: SVD review.**

For a matrix $A \in \mathbb{R}^{m \times n}$ with $\text{rank}(A) = \rho$:

1. Write the compact SVD: $A = U \Sigma V^T$ with shapes. (2 points)
2. If we truncate to rank $r < \rho$: $A_r = U_r \Sigma_r V_r^T$, what is $\text{rank}(A_r)$? (1 point)
3. What does the Eckart-Young theorem say about $A_r$? (2 points)

Reasoning is required.

### WRITE YOUR SOLUTION HERE ###

""" END OF THIS PART """

## Part 2 (10 points, non-coding task)

**Proof: GQA can be represented as MLA.**

Given GQA with groups $g = 1, \ldots, G$:
- $W^K_g \in \mathbb{R}^{D \times D_{qk}}$, $W^V_g \in \mathbb{R}^{D \times D_v}$

Prove that there exist MLA parameters $W^{DKV}, W^{UK}_h, W^{UV}_h$ such that for each head $h$ in group $g$:

$$W^{DKV} W^{UK}_h = W^K_g, \quad W^{DKV} W^{UV}_h = W^V_g$$

Your proof must use SVD. (10 points)

Reasoning is required.

### WRITE YOUR SOLUTION HERE ###

""" END OF THIS PART """

## Part 3 (10 points, coding task)

Implement `gqa_to_mla(W_K_groups, W_V_groups)` in NumPy.

Input: lists of $G$ key and value matrices.
Output: `W_DKV`, list of `W_UK_h` (one per head), list of `W_UV_h` (one per head).

Note: heads in the same group should get the same up-projection.

Use `np.linalg.svd`.

Reasoning is not required.

### WRITE YOUR SOLUTION HERE ###

""" END OF THIS PART """

## Part 4 (5 points, coding task)

Verify the conversion.

1. Create random GQA weights: $D=32, D_{qk}=8, D_v=8, G=4, H=8$ (2 heads per group).
2. Convert using your `gqa_to_mla`.
3. For each head $h$ in group $g$, assert $W^{DKV} @ W^{UK}_h \approx W^K_g$.
4. Print the MLA rank $r$ (should equal rank of the stacked matrix).

Reasoning is not required.

### WRITE YOUR SOLUTION HERE ###

""" END OF THIS PART """

## Part 5 (10 points, non-coding task)

**Counterexample: GQA $\subsetneq$ MLA.**

Construct an explicit MLA configuration that cannot be represented by any GQA with fewer than $H$ groups.

Requirements:
1. Specify $H, D, D_{qk}, r$. (2 points)
2. Specify $W^{DKV}$ and all $W^{UK}_h$. (3 points)
3. Compute all $W^K_h$ and show they are ALL distinct. (2 points)
4. Argue that no GQA with $G < H$ can represent this. (3 points)

Reasoning is required.

### WRITE YOUR SOLUTION HERE ###

""" END OF THIS PART """

## Part 6 (5 points, coding task)

Verify the counterexample numerically. Create the matrices from Part 5 and check distinctness.

Reasoning is not required.

### WRITE YOUR SOLUTION HERE ###

""" END OF THIS PART """

## Part 7 (10 points, non-coding task)

**Derivation: Reduced matrices for efficient MLA.**

Starting from MLA attention for head $h$:

$$\text{logits}_h = \frac{(X_1 W^Q_h)(X_2 W^{DKV} W^{UK}_h)^T}{\sqrt{D_{qk}}}$$

1. Expand $(X_2 W^{DKV} W^{UK}_h)^T$ as $(W^{UK}_h)^T (X_2 W^{DKV})^T$. (2 points)
2. Define $\hat{W}^Q_h = W^Q_h (W^{UK}_h)^T \in \mathbb{R}^{D \times r}$. Rewrite logits using $\hat{W}^Q_h$ and $C = X_2 W^{DKV}$. (3 points)
3. Show that the attention can be computed as $\text{logits}_h = \frac{(X_1 \hat{W}^Q_h) C^T}{\sqrt{D_{qk}}}$. (2 points)
4. What is the shape of $\hat{W}^Q_h$? Compare with $W^Q_h$. When is $\hat{W}^Q_h$ smaller? (3 points)

Reasoning is required.

### WRITE YOUR SOLUTION HERE ###

""" END OF THIS PART """

## Part 8 (10 points, coding task)

Implement reduced MLA using the absorbed query matrices.

```python
class MyMLAReduced(nn.Module):
    """MLA with reduced query — computes attention in latent space."""
```

Instead of storing $W^Q_h$ and $W^{UK}_h$ separately, precompute $\hat{W}^Q_h = W^Q_h (W^{UK}_h)^T$.

Forward steps:
1. $C = X W^{DKV}$ — compressed cache $(B, L, r)$
2. $\hat{Q} = X \hat{W}^Q$ — reduced query $(B, L, H \cdot r)$, reshape to $(B, H, L, r)$
3. $\text{logits}_h = \hat{Q}_h C^T / \sqrt{D_{qk}}$ — $(B, H, L, L)$
4. Compute values: $V = C W^{UV}$ and standard attention

Reasoning is not required.

### WRITE YOUR SOLUTION HERE ###

""" END OF THIS PART """

## Part 9 (10 points, coding task)

**Verify equivalence of standard MLA and reduced MLA.**

1. Create a standard `MyMLA` with $D=32, H=4, D_{qk}=8, D_v=8, r=16$.
2. Create a `MyMLAReduced` with the same dimensions.
3. Compute $\hat{W}^Q_h$ from the standard MLA's weights and set it in the reduced MLA.
4. Copy shared weights ($W^{DKV}, W^{UV}, W^O$).
5. Pass the same random input through both.
6. Assert outputs match within tolerance.

Reasoning is not required.

### WRITE YOUR SOLUTION HERE ###

""" END OF THIS PART """

## Part 10 (5 points, non-coding task)

In the reduced MLA:

1. What needs to be cached for KV-cache? Only $C$ or also $V$? (2 points)
2. If we ALSO absorb the value up-projection into the output (i.e., compute output directly from $\alpha C$ rather than $\alpha V$), what changes? (3 points)

Reasoning is required.

### WRITE YOUR SOLUTION HERE ###

""" END OF THIS PART """

## Part 11 (10 points, coding task)

**End-to-end: GQA $\to$ MLA $\to$ Reduced MLA.**

1. Create a GQA module with $D=32, H=4, G=2, D_{qk}=8, D_v=8$.
2. Convert GQA weights to MLA form (using SVD).
3. Build a standard MLA module with the converted weights.
4. Compute reduced query matrices $\hat{W}^Q_h$.
5. Build a reduced MLA module.
6. Pass the same input through all three: GQA, standard MLA, reduced MLA.
7. Assert all three outputs are equal (within tolerance).

This verifies the entire proof chain: GQA $\to$ MLA $\to$ Reduced MLA are equivalent.

Reasoning is not required.

### WRITE YOUR SOLUTION HERE ###

""" END OF THIS PART """

## Part 12 (10 points, non-coding task)

**Reflection: The proof chain.**

Summarize the complete proof chain in your own words:

1. MHA $\subseteq$ GQA: How? (2 points)
2. GQA $\subseteq$ MLA: The SVD argument in 3 sentences. (3 points)
3. GQA $\subsetneq$ MLA: The counterexample in 3 sentences. (3 points)
4. Practical implication: Why does this matter for deploying large language models? (2 points)

Reasoning is required.

### WRITE YOUR SOLUTION HERE ###

""" END OF THIS PART """